# 04 — Bias Audit

Load the trained ordinal age model, run it on the held-out test set, slice predictions by gender, ethnicity, age bucket, and intersectional group. Write `bias_report.json` and `bias_report.md` into `artifacts/`.

Run this after `slurm/train_age.sh` produces `artifacts/checkpoints/age_model_best.pt` on Surrey HPC and the checkpoint is pulled back to the laptop.

In [ ]:
# imports + path setup
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import random

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from torch.utils.data import DataLoader, Subset

from src.config import (ARTIFACTS_DIR, BATCH_SIZE, CHECKPOINTS_DIR,
                        ETHNICITY_LABELS, GENDER_LABELS, NUM_WORKERS, SEED)
from src.data.utkface_dataset import UTKFaceDataset
from src.data.transforms import get_inference_transform
from src.models.age_estimator import AgeEstimator
from src.models.ordinal_loss import OrdinalRegressionLoss
from src.training.evaluate import evaluate_model
from src.audit.bias_audit import run_bias_audit

sns.set_theme(style='whitegrid')

In [ ]:
# rebuild the same 80/10/10 split the training script used so the test set is identical
def split_indices(n, seed):
    indices = list(range(n))
    rng = random.Random(seed)
    rng.shuffle(indices)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)
    return (indices[:n_train], indices[n_train:n_train+n_val], indices[n_train+n_val:])

eval_ds_full = UTKFaceDataset(transform=get_inference_transform())
_, _, test_idx = split_indices(len(eval_ds_full), SEED)
test_ds = Subset(eval_ds_full, test_idx)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=False)

print(f'test set size: {len(test_ds):,}')

In [ ]:
# load the trained checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = CHECKPOINTS_DIR / 'age_model_best.pt'
print(f'loading checkpoint: {ckpt_path}')

model = AgeEstimator(pretrained=False).to(device)
checkpoint = torch.load(ckpt_path, map_location=device)
model.load_state_dict(checkpoint['model_state'])
model.eval()

print(f'checkpoint epoch: {checkpoint["epoch"]}')
print(f'checkpoint val MAE: {checkpoint["val_mae"]:.3f} years')

In [ ]:
# run the model on the test set, collect per-sample arrays for slicing
loss_fn = OrdinalRegressionLoss().to(device)
out = evaluate_model(model, test_loader, loss_fn, device, return_arrays=True)
arrays = out['arrays']

print(f'overall test MAE: {out["val_mae"]:.3f} years')
print(f'collected arrays: preds={arrays.preds.shape}, targets={arrays.targets.shape}')

In [ ]:
# run the audit — writes bias_report.json + bias_report.md into artifacts/
result = run_bias_audit(
    preds=arrays.preds,
    targets=arrays.targets,
    genders=arrays.genders,
    ethnicities=arrays.ethnicities,
    output_dir=ARTIFACTS_DIR,
)

print('overall:', result['overall'])
print('worst gap:', result['worst_gap'])
print(f'wrote {ARTIFACTS_DIR / "bias_report.json"}')
print(f'wrote {ARTIFACTS_DIR / "bias_report.md"}')

In [ ]:
# build a DataFrame from the group results so we can plot every axis
groups_df = pd.DataFrame(result['groups'])
groups_df.head(20)

In [ ]:
# bar charts — one per group_type axis. saves the figure used in README.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axis_map = {
    'gender': axes[0, 0],
    'ethnicity': axes[0, 1],
    'age_bucket': axes[1, 0],
    'gender_x_ethnicity': axes[1, 1],
}
for gtype, ax in axis_map.items():
    sub = groups_df[groups_df['group_type'] == gtype].sort_values('mae', ascending=False)
    if sub.empty:
        ax.set_title(f'{gtype} — no data')
        continue
    sns.barplot(data=sub, x='group_name', y='mae', ax=ax, color='steelblue')
    ax.axhline(result['overall']['mae'], color='red', linestyle='--', label='overall MAE')
    ax.set_title(f'MAE by {gtype}')
    ax.set_ylabel('MAE (years)')
    ax.tick_params(axis='x', rotation=30)
    ax.legend()

plt.tight_layout()
out_chart = ARTIFACTS_DIR / 'bias_chart.png'
plt.savefig(out_chart, dpi=130, bbox_inches='tight')
print(f'saved chart to {out_chart}')
plt.show()

## What to look for

Read the chart from worst to best:

1. **Worst group on each axis** — that is the group the model fails for most. Note the sample count alongside MAE; small groups have noisy MAEs.
2. **Worst-group gap** — the headline number. Anything above ~2 years on a real-world deployment is worth flagging in the compliance doc.
3. **Intersectional bottom-right panel** — bias usually hides in small intersectional cells (e.g. older Black women, young Asian men). Overall numbers can look fine while one cell is 2x worse.

These findings feed directly into `COMPLIANCE.md` (ITL ACCS notes) and the README's What I Learned section.